In [ ]:
folder = '.'

In [ ]:
# load
import json
import pathlib
import pandas as pd

folder = pathlib.Path(folder)
assert folder.exists()

# aggregate results into csv necessary
file_list = list(folder.glob('out*.json'))
dict_list = list()
for file in file_list:
    with open(file, 'r') as f:
        dict_list.append(json.load(f))
    
# load aggregated results
f_csv = folder / 'results.csv'
if f_csv.exists():
    df = pd.read_csv(f_csv, index_col=None)
else:
    df = pd.DataFrame()

# add in existing result
df = pd.concat((df, pd.DataFrame(dict_list)))

# round p_value to 14 decimal places (avoids floating point comparison failure)
df['p_val'] = df['p_val'].round(14)

# drop duplicates & check for conflicting results
df.drop_duplicates(inplace=True)
assert df.value_counts(subset=['p_val', 'seed', 'Analysis']).max() == 1
    
# overwrite csv with latest / greatest
df.to_csv(f_csv, index=False)

# delete json files (they're in csv)
for file in file_list:
    file.unlink()

In [ ]:
import numpy as np
from collections import defaultdict


# extract
pval_list = sorted(df['p_val'].unique())
seed_list = sorted(df['seed'].unique())

shape = len(seed_list), len(pval_list)
score_dict = defaultdict(lambda: np.full(shape=shape, fill_value=np.nan))

for _, row in df.iterrows():
    seed_idx = seed_list.index(row['seed'])
    pval_idx = pval_list.index(row['p_val'])
    
    for feat in ('f1', 'sens', 'spec'):
        score_dict[row['Analysis'], feat][seed_idx, pval_idx] = row[feat]

In [ ]:
# plot
import seaborn as sns
import matplotlib.pyplot as plt

sns.set()

fig, ax = plt.subplots(2, 3)

# plot top row
style_dict = {'AnalysisTFCE': {'color': 'r'}, 
              'AnalysisHRBA': {'color': 'b'}}
style_single = {'linewidth': .5,
                'zorder': 1,
                'label': '_nolegend_'}
style_mean = {'linewidth': 5,
              'zorder': 2,
              'label': '_nolegend_'}
for _ax, feat in zip(ax[0, :], ('f1', 'sens', 'spec')):
    plt.sca(_ax)
    for method, kwargs in style_dict.items():
        plt.plot(pval_list, score_dict[method, feat].T, **kwargs, **style_single)
        plt.plot(pval_list, np.nanmean(score_dict[method, feat], axis=0), **kwargs, **style_mean)

    plt.xlabel('p_val')
    plt.ylabel(feat)
    plt.xscale('log')

# plot bottom row
for _ax, feat in zip(ax[1, :], ('f1', 'sens', 'spec')):
    plt.sca(_ax)
    x = score_dict['AnalysisHRBA', feat] - score_dict['AnalysisTFCE', feat]
    plt.axhline([0], linewidth=2, color='k')
    plt.plot(pval_list, x.T, color='k', **style_single)
    plt.plot(pval_list, np.nanmean(x, axis=0), color='k', **style_mean)

    plt.xlabel('p_val')
    plt.ylabel(f'{feat}: HRBA - TFCE')
    plt.xscale('log')
    
# add legend in last plot of top row
plt.sca(ax[0, -1])
del style_single['label']
for method, kwargs in style_dict.items():
    plt.plot([], [], label=method[-4:], **kwargs, **style_single)
plt.legend()
    
fig.set_size_inches(10, 6)
fig.tight_layout()
fig.savefig(folder / 'HRBA_vs_TFCE.png', bbox_inches='tight')

# Count regions output by HRBA

In [ ]:
import cloudpickle as pickle
import gzip

for _, row in df[df['Analysis'] == 'AnalysisHRBA'].iterrows():
    # load it
    uuid = row['uuid']
    file_list = list(folder.glob(f'*{uuid}_detail*'))
    assert len(file_list) == 1
    file = file_list[0]
    
    with gzip.open(file, 'rb') as f:
        ana, effect = pickle.load(f)